In [16]:
import os
from pathlib import Path

from root import ROOT_PATH

os.chdir(ROOT_PATH)
import pandas

from src.consts import ANNOTATED_BASE_PATH

result_path = ANNOTATED_BASE_PATH / "4axis"

p = Path("/home/rsoleyma/projects/LabelStudioHelper/data/annotations_tables/2")
csvs = sorted(p.glob("*.json")).pop()

In [17]:
df_orig = pandas.read_json(csvs, orient="records")

nicknames = {
    "ramin": "🤨",
    "roosmouthaan": "🤨",
    "lalala": "🤨",
    "priscilag.costa":"🤨",
    "giulia.benati": "🤨",
    "xueyuan.liang": "🤨",
    "fulvia.calcagni": "🤨",
    "fulvia.calcagni_UAB": "🤨",
    "ariverca28":"🤨"
}

_R = "relevant-Relevant"
_U = "relevant-Uncertain"
_N = "relevant-Not relevant"


def set_nicknames(lst):
    return [nicknames.get(item, item) for item in lst]


def make_anon(col, icon: str = "🎯"):
    if col == "":
        return col
    return "".join([icon for _ in col])


def result_icon(i):
    if i == _R:
        return "🎯"
    elif i == _U:
        return "❓"
    elif i == _N:
        return "📭"
    return i


def merge(row):
    return (result_icon(_R) * len(row[_R]) +
            result_icon(_U) * len(row[_U]) +
            result_icon(_N) * len(row[_N]))


print(len(df_orig))
df = df_orig.copy()
df["post_text"] = df_orig["post_text"].apply(lambda x: x.replace("\n", ""))

# for col in df.columns:
#     df[col] = df[col].apply(lambda x: "" if pd.isna(x) else x)
# 
# # NICKNAMES
for col in df.columns[2:-1]:
    df[col] = df[col].apply(set_nicknames)

relevance_cols = ['relevant-Relevant',
                  'relevant-Not relevant',
                  'relevant-Uncertain']

landscape_cols = ['landscape-artificial surfaces',
                  'landscape-agricultural',
                  'landscape-Uncertain',
                  'landscape-forest and seminatural areas',
                  'landscape-wetlands',
                  'landscape-water bodies',
                  'landscape-not identifiable',
                  'landscape-ambigious']
dicho_cols = [
    'non_human-non-human',
    'non_human-human',
    'material-material',
    'material-ideel',
    'life-life',
    'life-mineral',
    'ideal_state-ideal_state',
    'ideal_state-alteration']

main_cols = relevance_cols + landscape_cols + dicho_cols

for col in relevance_cols:
    df[col + "_anon"] = df[col].apply(make_anon)


def nickname_notes(note):
    return {
        nicknames[k]: v for k, v in note.items()
    }


def anon_notes(note):
    return {
        f"‼️{idx}": v for idx, v in enumerate(note.values())
    }


df["notes"] = df["notes"].apply(nickname_notes)
df["notes_anon"] = df["notes"].apply(anon_notes)
df["merge"] = df.apply(merge, axis=1)

1001


In [18]:
df.head()

,task,post_text,relevant-Relevant,relevant-Not relevant,relevant-Uncertain,relevant-000,landscape-artificial surfaces,landscape-agricultural,landscape-Uncertain,landscape-forest and seminatural areas,...,life-000,ideal_state-ideal_state,ideal_state-alteration,ideal_state-000,notes,relevant-Relevant_anon,relevant-Not relevant_anon,relevant-Uncertain_anon,notes_anon,merge
0,1056,7 people followed me and 3 people unfollowed m...,[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[],[],[],[],[],...,"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",{},,🎯🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭📭
1,1057,Araujo's passes have been top notch today,[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[🤨],[],[],[],[],...,"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",{},,🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭
2,1058,What could be better than 3 rings in 1? #Gabr...,[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[🤨],[],[],[],[],...,"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",{},,🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭
3,1059,Aye everyone thanks for 400 follows… IN LESS T...,[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[🤨],[],[],[],[],...,"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",{},,🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭
4,1060,three solo wins in two days 😳,[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[🤨],[],[],[],[],...,"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",[],[],"[🤨, 🤨, 🤨, 🤨, 🤨, 🤨, 🤨]",{},,🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭


In [26]:
rel_col, rel_colN, notes_col = ['relevant-Relevant', 'relevant-Uncertain', "notes"]

mask = [bool(set(a) | set(b) | set(c)) for a, b, c in zip(df[rel_col], df[rel_colN], df[notes_col])]

res1 = df[mask][["post_text", "merge", "notes_anon"]]


def create_anon_html():
    """
    only merge and notes
    """
    res1.to_html(result_path / "anon.html")


def create_normal_html():
    df_orig[mask][["post_text", "notes"] + main_cols].to_html(result_path / "normal.html")

    df_add_nums = df_orig.copy()
    for col in main_cols:
        df_add_nums[f"{col}n"] = df_orig[col].apply(lambda e: len(e))
    df_add_nums.to_csv(result_path / "normal_nums.csv")
    
create_anon_html()
create_normal_html()


In [20]:
len(res1)

89

In [21]:
rel_col, rel_colN, notes_col = ['relevant-Relevant', 'relevant-Uncertain', "notes"]

res2 = df[mask][["post_text", "merge", "notes"] + main_cols]

In [22]:

### MAKE THIS NICER. APPLY TO NORMAL AS WELL
styled_df = res2.style.set_properties(**{
    'background-color': '#cfcbcb',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=relevance_cols).set_properties(**{
    'background-color': '#cfeb3b',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=["landscape-artificial surfaces", 'landscape-agricultural',
           'landscape-Uncertain',
           'landscape-forest and seminatural areas',
           'landscape-wetlands',
           'landscape-water bodies',
           'landscape-not identifiable',
           'landscape-ambigious']).set_properties(**{
    'background-color': '#afebbb',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=['non_human-non-human',
           'non_human-human']).set_properties(**{
    'background-color': '#ff2b5b',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=['material-material',
           'material-ideel']).set_properties(**{
    'background-color': '#0febeb',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=['life-life',
           'life-mineral']).set_properties(**{
    'background-color': '#afeb7b',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=[
    'ideal_state-ideal_state',
    'ideal_state-alteration'])

# Combine with CSS
html = styled_df.to_html()
css = '''
<style>
.custom-class {
    font-weight: bold;
    padding: 10px;
}
</style>
'''
complete_html = css + html
(result_path / "full-icons.html").write_text(complete_html)
print((ANNOTATED_BASE_PATH / "full-icons.html").absolute())

/home/rsoleyma/projects/twitter-stream-unpacker/data/annotated/full-icons.html


In [23]:
import markdown


def get_task_html(row, emoji: bool = False):
    def mod_userlist(lst: list[str]) -> str:
        if not lst:
            return "✖️"
        if emoji:
            return "".join([nicknames[c] for c in lst])
        else:
            return ", ".join(lst)

    s = []
    s.append("----")

    # Kia ora. Let me introduce myself! I'm carol. let me show you what fine food I find in the world. Do you know where I am? I am in South Korea🇰🇷. The weather now is ☁️🌤🌧☁️☁️🌧.
    #
    if "Kia ora" in row["post_text"]:
        print()
    for line in row["post_text"].split("\n"):
        if line:
            s.append(f"## {line}")
    s.append(f"### {row['merge']}")

    for u in relevance_cols:
        if row[u]:
            s.append(f"__{u} {result_icon(u)}:__ {mod_userlist(row[u])}")
    # print(",".join(row.keys()))
    s.append("### Landscape")
    for u in ['landscape-artificial surfaces',
              'landscape-agricultural',
              'landscape-Uncertain',
              'landscape-forest and seminatural areas',
              'landscape-wetlands',
              'landscape-water bodies',
              'landscape-not identifiable',
              'landscape-ambigious']:
        if row[u]:
            s.append(f"__{u}:__ {mod_userlist(row[u])}")

    s.append("### 4 Axis")
    ## here get the proper pandas fraction and create a normal table.
    for u1, u2 in [['non_human-non-human',
                    'non_human-human'],
                   ['material-material',
                    'material-ideel'],
                   ['life-life',
                    'life-mineral'],
                   ['ideal_state-ideal_state',
                    'ideal_state-alteration']]:
        #pd.DataFrame([u1,)
        si = u1.index("-")
        name = u1[:si]
        u1n = u1[si + 1:]
        u2n = u2[si + 1:]
        alert = 1 if u1n and u2n else ""

        s.append(f"__{name} {alert}::: {u1n}:__ {mod_userlist(row[u1])} __{u2n}:__ {mod_userlist(row[u2])}")
    if row["notes"]:
        s.append("### Notes")
        for coder, note in row["notes"].items():
            s.append(f"__{coder}:__ {note}")
    return "\n\n".join(s)


# 
df_orig["merge"] = res2["merge"]

# with open(result_path / "easy-emo.html", "w", encoding="utf-8") as f:
#     all_lines = []
#     for row in df_orig[mask].to_dict('records'):
#         all_lines.append(get_task_html(row, True))
#     res = "\n\n".join(all_lines)
#     html = markdown.markdown(res)
#     f.write(html)

with open(result_path / "easy.html", "w", encoding="utf-8") as f:
    all_lines = []
    for row in df_orig[mask].to_dict('records'):
        all_lines.append(get_task_html(row, False))
    res = "\n\n".join(all_lines)
    html = markdown.markdown(res)
    f.write(html)

In [24]:
# USE
res3 = res2.copy()
include = []
for col in main_cols:
    res3[f"{col}n"] = res2[col].apply(lambda e: len(e))
res2.to_csv(result_path / "res2.csv")

res3.to_csv(result_path / "res3.csv")

In [11]:
result_path

PosixPath('data/annotated/4axis')

In [12]:
# bring back the column that has the original tweet url

In [20]:
import json

p = Path("/home/rsoleyma/projects/language_stuff/twitter-text-1-disagreements.json")
claude = pandas.json_normalize(json.loads(p.read_text())["results"])

In [21]:
claude.head()

,id,content,model,role,stop_reason,stop_sequence,type,messages,usage.input_tokens,usage.output_tokens
0,msg_01Q16jsT8kuobsJgdr6La6Er,"[{'text': 'not-relevant ENTERTAINMENT_FOCUS', ...",claude-3-5-sonnet-20241022,assistant,max_tokens,None,message,"[{'role': 'user', 'content': 'Is Slut Pop by K...",162,10
1,msg_01Pok6eMDTngb6fjM3f5QKid,"[{'text': 'not-relevant PERSONAL_DISCOMFORT', ...",claude-3-5-sonnet-20241022,assistant,max_tokens,None,message,"[{'role': 'user', 'content': 'All that food I ...",139,10
2,msg_0125MYMue6VDyNYCxf3xQXcv,"[{'text': 'not-relevant FICTIONAL_NARRATIVE', ...",claude-3-5-sonnet-20241022,assistant,max_tokens,None,message,"[{'role': 'user', 'content': '""Man I wish when...",174,10
3,msg_01SFaugpXZ7GbE3S3ag9L6w5,"[{'text': 'not-relevant DOMESTIC_PETS', 'type'...",claude-3-5-sonnet-20241022,assistant,max_tokens,None,message,"[{'role': 'user', 'content': 'Dog/Dog with tur...",156,10
4,msg_01Toz1LH33iqWpK87Pz85yxh,"[{'text': 'not-relevant CASUAL_OBSERVATION', '...",claude-3-5-sonnet-20241022,assistant,max_tokens,None,message,"[{'role': 'user', 'content': 'Morning worm 🪱 h...",146,10


In [36]:
claude["message"] = claude["messages"].apply(lambda k: k[0].get("content"))
claude["text"] = claude["content"].apply(lambda k: k[0].get("text"))

In [39]:
claude[["text", "message"]]

prediction_relevant = "⬆️"
prediction_not_relevant = "⬇️"

,text,message
0,not-relevant ENTERTAINMENT_FOCUS,Is Slut Pop by Kim Petras going to be summer a...
1,not-relevant PERSONAL_DISCOMFORT,All that food I ate earlier is kicking inI am ...
2,not-relevant FICTIONAL_NARRATIVE,"""Man I wish when Asuna trained tirelessly the ..."
3,not-relevant DOMESTIC_PETS,Dog/Dog with turkey bacon on snoot/ Dog eating...
4,not-relevant CASUAL_OBSERVATION,Morning worm 🪱 https://t.co/CYwC8FWImf
5,not-relevant PERSONAL_INTRODUCTION,Kia ora. Let me introduce myself! I'm carol. l...
6,not-relevant METAPHORICAL_LANGUAGE,"""Month's home, water is a sweet home."""
7,relevant SPIRITUAL_REFLECTION,"""i give thanks for this world as a place to le..."
8,relevant ENVIRONMENTAL_POLICY,Save the date for the launch of the OECD Globa...
9,not-relevant PLANT_MAINTENANCE,Who said succulents were the easiest plant to ...


In [31]:
import pandas as pd
from jsonpath_ng import parse


def extract_with_jsonpath(df, column, jsonpath_expr):
    """
    Extract data from dictionary cells in a DataFrame using JSONPath expressions.
    
    Parameters:
    df (pandas.DataFrame): Input DataFrame
    column (str): Name of the column containing dictionaries
    jsonpath_expr (str): JSONPath expression to extract data
    
    Returns:
    pandas.Series: Series containing extracted values
    """
    # Compile the JSONPath expression
    jsonpath_parser = parse(jsonpath_expr)
    print(jsonpath_parser)

    def extract_value(cell):
        if pd.isna(cell) or not isinstance(cell, dict):
            return None
        matches = [match.value for match in jsonpath_parser.find(cell)]
        return matches[0] if matches else None

    return df[column].apply(extract_value)


# '$[0].text'       # first element's text
# '$..text'         # any text field anywhere
# '$.[0].text'      # alternate syntax
# '$.*[0].text'     # first element of any array's text
extract_with_jsonpath(claude, "content", '$.*[0].text')

$.'*'.[0].text


0     None
1     None
2     None
3     None
4     None
5     None
6     None
7     None
8     None
9     None
10    None
11    None
12    None
13    None
14    None
15    None
16    None
17    None
18    None
19    None
Name: content, dtype: object

In [42]:
import pandas as pd


import json
from pathlib import Path
p = Path("/home/rsoleyma/projects/language_stuff/twitter-text-1-disagreements.json")
results = json.loads(p.read_text())["results"]
for result in results:
    result["content"] = result["content"][0]
    result["messages"] = result["messages"][0]
    
    
claude = pd.json_normalize(results)
sentence_map  ={}

claude["post_text"] = claude["messages.content"]
    #claude["messages.content"] = claude["id"]
df_orig.merge(claude, on="post_text")
    # df_orig["ai"] = df_orig.loc[df['post_text'] == claude["messages.content"]]["messages.content"]
print(claude.columns, df_orig.columns) 
def create_normal_html2():
    #df_orig[mask][["post_text", "notes"] + main_cols].to_html(result_path / "normal.html")

    df_add_nums = df_orig.copy()
    
    for col in main_cols:
        df_add_nums[f"{col}n"] = df_orig[col].apply(lambda e: len(e))
    df_add_nums.to_csv(result_path / "normal_nums.csv")
    
    with open(result_path / "easy.html", "w", encoding="utf-8") as f:
        all_lines = []
        for row in df_orig[mask].to_dict('records'):
            all_lines.append(get_task_html(row, False))
        res = "\n\n".join(all_lines)
        html = markdown.markdown(res)
        f.write(html)
        
create_normal_html2()

Index(['id', 'model', 'role', 'stop_reason', 'stop_sequence', 'type',
       'content.text', 'content.type', 'usage.input_tokens',
       'usage.output_tokens', 'messages.role', 'messages.content',
       'post_text'],
      dtype='object') Index(['task', 'post_text', 'relevant-Relevant', 'relevant-Not relevant',
       'relevant-Uncertain', 'relevant-000', 'landscape-artificial surfaces',
       'landscape-agricultural', 'landscape-Uncertain',
       'landscape-forest and seminatural areas', 'landscape-wetlands',
       'landscape-water bodies', 'landscape-not identifiable',
       'landscape-ambigious', 'landscape-000', 'non_human-non-human',
       'non_human-human', 'non_human-000', 'material-material',
       'material-ideel', 'material-000', 'life-life', 'life-mineral',
       'life-000', 'ideal_state-ideal_state', 'ideal_state-alteration',
       'ideal_state-000', 'notes', 'merge'],
      dtype='object')



{'id': 'msg_01Q16jsT8kuobsJgdr6La6Er',
 'content': [{'text': 'not-relevant ENTERTAINMENT_FOCUS', 'type': 'text'}],
 'model': 'claude-3-5-sonnet-20241022',
 'role': 'assistant',
 'stop_reason': 'max_tokens',
 'stop_sequence': None,
 'type': 'message',
 'usage': {'input_tokens': 162, 'output_tokens': 10},
 'messages': [{'role': 'user',
   'content': 'Is Slut Pop by Kim Petras going to be summer anthem for me? I think soooo https://t.co/0GG2AXCL2c'}]}

In [33]:
len(claude)

20

In [51]:
sorted(set(df_orig["post_text"]))

['"Don\'t be a stick in the mud, now!" #PsycheBot',
 '"Excellence in the gradual result of always striving to do better." Pat Riley #quotes https://t.co/khrqKr9T74',
 '"I don\'t like abilities in Destiny"\n\nthen https://t.co/p5yBVlycQu',
 '"I\'m so sad" deserve tanginamo bading.',
 '"Maybe he didn\'t feel loved at West Ham."\n\n"He\'s gone on to a totally different level now."\n\n41 goals in 51 games. Sébastien Haller\'s form at Ajax has been sensational! 🔥\n\n@seemajaswal, @themichaelowen and @chris_sutton73 discuss the #UCL\'s top scorer this season 📈 https://t.co/L1Zp0MaGFd',
 '"The Adventures of Rocky Jordan" is up next, with "City of Baksheesh".\n\nOriginally broadcast in 1950.\n\nTune in at https://t.co/aoukajHnq7\n\n#OldTimeRadio #NowPlaying #OTR',
 '## for jimin❗',
 '#15March_GaribdasJiBodhDiwas\nUnder the guidance of JagatGuru Tatvdarshi Sant Rampal Ji Maharaj, on the occasion of Bodh Diwas of respected Sant Garib Das Ji Maharaj,\nWatch live telecast of a special program on S

In [52]:
df_orig["post_text"] = 
for a in set(claude["post_text"]):
    if not a in set(df_orig["post_text"]):
        for t in df_orig["post_text"]:
            print(.startswith(a[:10]))

AttributeError: 'Series' object has no attribute 'startswith'